# Filtering DataFrames

**Learning Objectives:**
- Understand boolean indexing and how to create boolean masks
- Filter DataFrames using comparison operators (`==`, `>`, `<`, etc.)
- Use pandas methods like `.between()` and `.isin()` for filtering
- Combine multiple conditions using `&` (AND) and `|` (OR)
- Apply the `.query()` method as a modern alternative
- Understand when to use `.loc[]` vs direct bracket indexing

## 1. Introduction to filtering

Filtering is one of the most common operations when working with data. It allows you to extract specific rows from a DataFrame based on conditions.

<!-- INSTRUCTOR: Emphasize real-world scenarios - finding customers from a specific region, products above a price threshold, etc. -->

📚 **Concept**: Filtering lets you answer questions like "Which athletes won gold medals?" or "What happened in the 2012 Olympics?" by creating subsets of your data.

In [ ]:
import pandas as pd

In [ ]:
# Load Olympic Summer Games medal data
df = pd.read_csv("summer.csv")

In [ ]:
# Display first few rows to understand the data
df.head()

In [ ]:
# Check the shape and basic info
print(f"Total medals in dataset: {len(df)}")
df.info()

## 2. Boolean indexing basics

📚 **Concept**: Boolean indexing uses a Series of `True`/`False` values to select rows. When `True`, the row is included; when `False`, it's excluded.

<!-- INSTRUCTOR: Draw analogy to a checklist - only items with checkmarks are included -->

### 2.1 Creating a boolean mask

Let's start by finding all medals won by Swedish athletes.

In [ ]:
# Create a comparison - returns True/False for each row
df["Country"] == "SWE"

💡 **Tip**: This creates a boolean Series - the same length as the DataFrame, with `True` where the condition is met.

In [ ]:
# Count how many True and False values we have
(df["Country"] == "SWE").value_counts()

### 2.2 Using the mask to filter

In [ ]:
# Store the boolean Series in a variable (commonly called a "mask")
mask = df["Country"] == "SWE"

In [ ]:
# Use .loc[] to filter with the mask
df.loc[mask]

✅ **Key Takeaway**: A mask is just a boolean Series used to filter rows. Give it a descriptive name like `mask_sweden` or `is_gold_medal`.

### 2.3 Alternative syntax without .loc[]

In [ ]:
# You can also filter using bracket notation directly
df[mask]

💡 **Tip**: Both `df.loc[mask]` and `df[mask]` work for simple filtering. Use `.loc[]` when you also want to select specific columns, e.g., `df.loc[mask, ["Athlete", "Medal"]]`.

In [ ]:
# Example: Filter rows AND select specific columns
df.loc[mask, ["Year", "Athlete", "Event", "Medal"]].head()

### ✅ Quick Check

Before proceeding, can you:
- [ ] Explain what a boolean mask is?
- [ ] Create a comparison that returns True/False values?
- [ ] Use that mask to filter a DataFrame?

## 3. Comparison operators

You can use any comparison operator to create boolean masks:

| Operator | Meaning | Example |
|----------|---------|----------|
| `==` | Equal to | `df["Year"] == 2012` |
| `!=` | Not equal to | `df["Medal"] != "Bronze"` |
| `>` | Greater than | `df["Year"] > 2000` |
| `<` | Less than | `df["Year"] < 1920` |
| `>=` | Greater than or equal | `df["Year"] >= 2000` |
| `<=` | Less than or equal | `df["Year"] <= 1900` |

### 3.1 Filtering with equality

In [ ]:
# Find all medals from the first modern Olympics (1896)
df.loc[df["Year"] == 1896]

### 3.2 Filtering with greater than / less than

In [ ]:
# Find all medals from 21st century Olympics
df.loc[df["Year"] > 2000]

💡 **Tip**: For numeric columns, `>`, `<`, `>=`, and `<=` work great. For text columns, stick with `==` and `!=`.

## 4. Pandas filtering methods

📚 **Concept**: Pandas provides special methods that make certain filtering tasks easier and more readable.

### 4.1 Using .between() for ranges

In [ ]:
# Find medals from the 1990s (1992, 1996, 2000 Olympics)
df.loc[df["Year"].between(1990, 2000)]

✅ **Key Takeaway**: `.between()` is cleaner than writing `(df["Year"] >= 1990) & (df["Year"] <= 2000)`. It's inclusive by default (includes both endpoints).

### 4.2 Using .isin() for multiple values

In [ ]:
# Find medals from first and most recent Olympics in dataset
df.loc[df["Year"].isin([1896, 2012])]

💡 **Tip**: `.isin()` accepts a list of values. It's much cleaner than writing `(df["Year"] == 1896) | (df["Year"] == 2012)`.

In [ ]:
# Example: Find medals from London or Sydney Olympics
df.loc[df["City"].isin(["London", "Sydney"])]

In [ ]:
# Verify our filter worked correctly
df.loc[df["City"].isin(["London", "Sydney"])]["City"].value_counts()

### ✅ Quick Check

Before proceeding, can you:
- [ ] Use `.between()` to find rows in a numeric range?
- [ ] Use `.isin()` to filter for multiple specific values?
- [ ] Explain when `.isin()` is better than multiple `==` comparisons?

## 5. Combining conditions with & and |

📚 **Concept**: Real-world filtering often requires multiple conditions. Use `&` (AND) when all conditions must be true, and `|` (OR) when at least one condition must be true.

<!-- INSTRUCTOR: Emphasize the importance of parentheses - operator precedence issue is very common mistake -->

### 5.1 The & operator (AND)

Use `&` when you want rows that match ALL conditions.

In [ ]:
# Find Swedish women's medals
df.loc[(df["Country"] == "SWE") & (df["Gender"] == "Women")]

⚠️ **Warning**: You MUST use parentheses around each condition when using `&` or `|`. This is due to Python's operator precedence.

In [ ]:
# ⚠️ INTENTIONAL ERROR - This will fail!
# df.loc[df["Country"] == "SWE" & df["Gender"] == "Women"]
# Error: Operator precedence causes & to evaluate first, creating invalid comparison

🐛 **Debug**: The error `TypeError: Cannot perform 'rand_' with a dtyped [object] array` means you forgot parentheses around your conditions.

### 5.2 Combining three or more conditions

In [ ]:
# Find Swedish women's medals from 21st century
df.loc[(df["Country"] == "SWE") & 
       (df["Gender"] == "Women") & 
       (df["Year"] > 2000)]

💡 **Tip**: When combining many conditions, break them across multiple lines for readability. Each condition still needs its own parentheses.

### 5.3 Using separate masks for clarity

In [ ]:
# Create descriptive masks for each condition
is_sweden = df["Country"] == "SWE"
is_women = df["Gender"] == "Women"
is_recent = df["Year"] > 2000

# Combine them with clear names
df.loc[is_sweden & is_women & is_recent]

✅ **Key Takeaway**: Using separate named masks makes your code more readable and easier to debug.

### 5.4 The | operator (OR)

Use `|` when you want rows that match AT LEAST ONE condition.

In [ ]:
# Find medals from either Scandinavian countries
is_nordic = (df["Country"] == "SWE") | (df["Country"] == "NOR") | (df["Country"] == "DEN")
df.loc[is_nordic]

💡 **Tip**: When using `|` with the same column, `.isin()` is much cleaner:

In [ ]:
# Same result, cleaner syntax
df.loc[df["Country"].isin(["SWE", "NOR", "DEN"])]

### 5.5 The ~ operator (NOT)

Use `~` to invert a boolean mask (flip `True` to `False` and vice versa).

In [ ]:
# Find all medals that are NOT for men
df.loc[~(df["Gender"] == "Men")]

💡 **Tip**: You can also use `!=` for simple negation: `df["Gender"] != "Men"` is equivalent to `~(df["Gender"] == "Men")`.

In [ ]:
# More complex example: Nordic women's medals
is_nordic = df["Country"].isin(["SWE", "NOR", "DEN"])
is_not_men = ~(df["Gender"] == "Men")

df.loc[is_nordic & is_not_men]

### ✅ Quick Check

Before proceeding, can you:
- [ ] Combine two conditions using `&` (AND)?
- [ ] Explain why parentheses are required around each condition?
- [ ] Use `|` (OR) for alternative conditions?
- [ ] Use `~` (NOT) to invert a boolean mask?

## 6. The .query() method

📚 **Concept**: The `.query()` method provides a more readable way to filter DataFrames using string expressions.

<!-- INSTRUCTOR: Mention that query() is particularly useful when conditions are built dynamically or when working with many columns -->

### 6.1 Basic query syntax

In [ ]:
# Traditional filtering
df.loc[df["Year"] > 2000]

In [ ]:
# Same filter using .query()
df.query("Year > 2000")

💡 **Tip**: Notice with `.query()` you don't need quotes around column names (unless they have spaces), and you don't need to repeat the DataFrame name.

### 6.2 Combining conditions in queries

In [ ]:
# Use 'and' instead of &
df.query("Country == 'SWE' and Gender == 'Women'")

In [ ]:
# Use 'or' instead of |
df.query("Country == 'SWE' or Country == 'NOR'")

✅ **Key Takeaway**: In `.query()`, use `and`/`or`/`not` (words) instead of `&`/`|`/`~` (symbols), and you don't need parentheses around each condition.

### 6.3 Using variables in queries

In [ ]:
# Reference Python variables with @ symbol
year_threshold = 2000
df.query("Year > @year_threshold")

In [ ]:
# Works with lists too
nordic_countries = ["SWE", "NOR", "DEN"]
df.query("Country in @nordic_countries")

💡 **Tip**: The `@` symbol tells `.query()` to look for a Python variable rather than a column name.

### 6.4 When to use .query() vs boolean indexing

**Use `.query()` when:**
- You have many conditions (improves readability)
- Building filters dynamically from user input
- Working with very large datasets (can be faster)

**Use boolean indexing when:**
- Simple one or two condition filters
- Need more complex logic that `.query()` doesn't support
- Want better autocomplete/IDE support (column names in quotes don't autocomplete)

## 7. Practical examples and patterns

Let's apply what we've learned to answer real questions about the data.

### 7.1 Finding specific medals

In [ ]:
# Which sports awarded medals in the first Olympics?
first_olympics = df.query("Year == 1896")
first_olympics["Sport"].unique()

### 7.2 Counting with filters

In [ ]:
# How many gold medals has Sweden won?
sweden_gold = df.query("Country == 'SWE' and Medal == 'Gold'")
print(f"Sweden has won {len(sweden_gold)} gold medals")

### 7.3 Comparing subsets

In [ ]:
# Compare medal distribution for men vs women in recent Olympics
recent = df.query("Year >= 2000")

print("Recent Olympics medals by gender:")
print(recent["Gender"].value_counts())

### 7.4 Chaining filters with method chaining

In [ ]:
# Find top 5 countries with most gold medals in Swimming since 2000
(df.query("Year >= 2000 and Discipline == 'Swimming' and Medal == 'Gold'")
   ["Country"]
   .value_counts()
   .head())

✅ **Key Takeaway**: Filtering is often the first step in a data analysis pipeline, followed by aggregation, sorting, or visualization.

## 8. Summary

In this notebook, you learned:

1. **Boolean indexing** - Creating masks with True/False values to filter rows
2. **Comparison operators** - Using `==`, `>`, `<`, etc. to create conditions
3. **Pandas methods** - `.between()` for ranges, `.isin()` for multiple values
4. **Combining conditions** - Using `&` (AND), `|` (OR), and `~` (NOT) with parentheses
5. **The .query() method** - A cleaner alternative using string expressions
6. **Practical patterns** - Real-world examples of filtering for data analysis

<!-- INSTRUCTOR: Emphasize practicing with their own datasets - filtering is a foundational skill -->

## Quick Reference: Filtering in pandas

### Syntax

```python
# Basic filtering
df.loc[df["column"] == value]
df[df["column"] == value]  # Alternative

# Multiple conditions (requires parentheses!)
df.loc[(df["col1"] == value1) & (df["col2"] > value2)]

# Using .query()
df.query("col1 == value1 and col2 > value2")
```

### Common Operations

| Operation | Code | Example |
|-----------|------|----------|
| Equality | `==` | `df["Country"] == "USA"` |
| Not equal | `!=` | `df["Medal"] != "Bronze"` |
| Range | `.between()` | `df["Year"].between(1990, 2000)` |
| Multiple values | `.isin()` | `df["City"].isin(["London", "Paris"])` |
| AND | `&` | `(condition1) & (condition2)` |
| OR | `\|` | `(condition1) \| (condition2)` |
| NOT | `~` | `~(condition)` |

### Common Errors

| Error | Cause | Fix |
|-------|-------|-----|
| `TypeError: 'rand_'` | Missing parentheses with `&` | Add `()` around each condition |
| `KeyError` | Column name typo | Check exact column name with `df.columns` |
| `ValueError` | Wrong data type comparison | Convert types: `df["col"].astype(int)` |